# Causal Diagnostics Report

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
from pathlib import Path
import csv

from docx import Document
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor


ROOT = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
DIAG = ROOT / "Vivacity_full_day_cutoff_20260526" / "modelling_ready" / "causal_diagnostics"
OUT = ROOT / "dissertation_causal_diagnostics_report_vivacity.docx"


def read_csv(path):
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)


def set_cell_margins(cell, top=80, start=100, bottom=80, end=100):
    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()
    tc_mar = tc_pr.first_child_found_in("w:tcMar")
    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)
    for m, v in [("top", top), ("start", start), ("bottom", bottom), ("end", end)]:
        node = tc_mar.find(qn(f"w:{m}"))
        if node is None:
            node = OxmlElement(f"w:{m}")
            tc_mar.append(node)
        node.set(qn("w:w"), str(v))
        node.set(qn("w:type"), "dxa")


def set_table_borders(table):
    tbl_pr = table._tbl.tblPr
    borders = tbl_pr.first_child_found_in("w:tblBorders")
    if borders is None:
        borders = OxmlElement("w:tblBorders")
        tbl_pr.append(borders)
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        elem = borders.find(qn(f"w:{edge}"))
        if elem is None:
            elem = OxmlElement(f"w:{edge}")
            borders.append(elem)
        elem.set(qn("w:val"), "single")
        elem.set(qn("w:sz"), "4")
        elem.set(qn("w:space"), "0")
        elem.set(qn("w:color"), "DADCE0")


def add_run(paragraph, text, bold=False, italic=False, size=11, color="000000"):
    run = paragraph.add_run(text)
    run.bold = bold
    run.italic = italic
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    run.font.size = Pt(size)
    run.font.color.rgb = RGBColor.from_string(color)
    return run


def add_para(doc, text):
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(8)
    p.paragraph_format.line_spacing = 1.15
    add_run(p, text)
    return p


def add_heading(doc, text, level=1):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(18 if level == 1 else 12)
    p.paragraph_format.space_after = Pt(6)
    add_run(p, text, size=19 if level == 1 else 14)
    return p


def add_bullets(doc, items):
    for item in items:
        p = doc.add_paragraph(style="List Bullet")
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.15
        add_run(p, item)


def fmt_pct(value):
    if value in ("", None):
        return "NA"
    return f"{float(value):+.2f}%"


def fmt_p(value):
    if value in ("", None):
        return "NA"
    v = float(value)
    return "<0.001" if v < 0.001 else f"{v:.3f}"


def add_pretrend_table(doc, rows):
    table = doc.add_table(rows=1, cols=6)
    table.autofit = False
    widths = [0.65, 1.15, 1.05, 0.7, 1.45, 1.3]
    for i, width in enumerate(widths):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = ["Scheme", "Outcome", "Pre-period", "p", "Slope diff.", "Decision"]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=8.5)

    for row in rows:
        decision = "warning" if row["diagnostic"] == "pretrend_warning" else "no warning"
        values = [
            row["scheme_id"],
            row["outcome_label"],
            f"{row['first_pre_week']} to {row['last_pre_week']}",
            fmt_p(row["p_value"]),
            f"{fmt_pct(row['percent_change_per_week'])} per week",
            decision,
        ]
        cells = table.add_row().cells
        for i, value in enumerate(values):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            if i in [0, 3, 5]:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            add_run(p, value, size=8.2)


def add_event_flag_table(doc, rows):
    flags = [
        r for r in rows
        if r["diagnostic"] in ("pre_event_warning", "post_event_flagged")
    ]
    table = doc.add_table(rows=1, cols=6)
    table.autofit = False
    widths = [0.65, 1.15, 1.15, 0.8, 1.0, 1.65]
    for i, width in enumerate(widths):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = ["Scheme", "Outcome", "Event bin", "Period", "Effect", "Diagnostic"]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=8.5)

    for row in flags:
        values = [
            row["scheme_id"],
            row["outcome_label"],
            row["event_bin"],
            row["period"],
            f"{fmt_pct(row['percent_change'])}; p={fmt_p(row['p_value'])}",
            row["diagnostic"].replace("_", " "),
        ]
        cells = table.add_row().cells
        for i, value in enumerate(values):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            if i in [0, 3]:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            add_run(p, value, size=8.2)


def add_decision_table(doc, summary_rows):
    table = doc.add_table(rows=1, cols=4)
    table.autofit = False
    widths = [0.65, 1.25, 1.4, 3.0]
    for i, width in enumerate(widths):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = ["Scheme", "Pre-trend", "Event study", "Final credibility decision"]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=8.5)

    decisions = {
        "12d": "Not credible for strong causal wording. Use as exploratory only because active travel and pedestrian pre-trends are already divergent.",
        "12f": "Partly credible but still exploratory. Linear pre-trends pass, but event-study active-travel pre-period warning and small sample prevent definitive causal claims.",
        "13": "Partly credible but still exploratory. Linear pre-trends pass, but cyclist pre-event warning and control-verification uncertainty remain.",
    }
    for row in summary_rows:
        values = [
            row["scheme_id"],
            f"{row['pretrend_warnings']} warnings",
            f"{row['pre_event_warnings']} pre warnings; {row['post_event_flags']} post flags",
            decisions.get(row["scheme_id"], row["causal_readiness"]),
        ]
        cells = table.add_row().cells
        for i, value in enumerate(values):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            if i == 0:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            add_run(p, value, size=8.2)


def add_plot(doc, filename, caption):
    path = DIAG / "plots" / filename
    if not path.exists():
        return
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(12)
    p.paragraph_format.space_after = Pt(4)
    add_run(p, caption, bold=True, size=10)
    doc.add_picture(str(path), width=Inches(6.3))


def build_doc():
    pretrend = read_csv(DIAG / "vivacity_pretrend_diagnostics.csv")
    event = read_csv(DIAG / "vivacity_event_study_coefficients.csv")
    summary = read_csv(DIAG / "vivacity_causal_diagnostic_summary.csv")

    doc = Document()
    section = doc.sections[0]
    section.top_margin = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin = Inches(1)
    section.right_margin = Inches(1)

    styles = doc.styles
    styles["Normal"].font.name = "Arial"
    styles["Normal"]._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    styles["Normal"].font.size = Pt(11)

    title = doc.add_paragraph()
    title.paragraph_format.space_after = Pt(3)
    add_run(title, "Causal Diagnostics Report: Vivacity Active Travel Models", size=25)

    subtitle = doc.add_paragraph()
    subtitle.paragraph_format.space_after = Pt(12)
    add_run(
        subtitle,
        "Pre-trend tests, event-study diagnostics, and control credibility decision",
        size=11,
        color="555555",
    )

    add_heading(doc, "Purpose", 1)
    add_para(
        doc,
        "This report tests whether the current matched-control Vivacity modelling design is credible enough to support stronger causal claims. It should be read as a diagnostic companion to the exploratory treated-control models.",
    )

    add_heading(doc, "Overall Decision", 1)
    add_para(
        doc,
        "The current design should remain exploratory. The diagnostics strengthen transparency, but they do not justify describing the model as a final strong causal impact estimate. The most defensible label is: exploratory matched-control interrupted time-series analysis with pre-trend and event-study diagnostics.",
    )
    add_decision_table(doc, summary)

    add_heading(doc, "Pre-Trend Tests", 1)
    add_para(
        doc,
        "The pre-trend test uses only pre-intervention observations. The key term is the treated-control difference in pre-intervention slope. A statistically flagged term indicates that treated and control countlines were already moving differently before the scheme month.",
    )
    add_pretrend_table(doc, pretrend)

    add_heading(doc, "Event-Study Flags", 1)
    add_para(
        doc,
        "The event-study diagnostic estimates treated-control differences in relative-week bins, using the four weeks before intervention as the reference period. Pre-period warnings weaken causal credibility; post-period flags identify possible divergence after intervention but must be interpreted only after checking pre-period behaviour.",
    )
    add_event_flag_table(doc, event)

    add_heading(doc, "Interpretation for Dissertation", 1)
    add_bullets(
        doc,
        [
            "12d should not be used for strong causal wording because active travel and pedestrian pre-trends fail the diagnostic screen.",
            "12f is the strongest exploratory comparison, but it still has an active-travel pre-event warning and a small countline pool.",
            "13 has better pre-period coverage, but cyclist event-study diagnostics show a pre-period warning.",
            "The methodology chapter should include these diagnostics as robustness checks and should explicitly state that they support cautious exploratory interpretation.",
        ],
    )

    add_heading(doc, "Event-Study Plots", 1)
    add_plot(
        doc,
        "event_study_active_per_observed_day.png",
        "Figure 1. Active travel event-study treated-control differences.",
    )
    add_plot(
        doc,
        "event_study_pedestrian_per_observed_day.png",
        "Figure 2. Pedestrian event-study treated-control differences.",
    )
    add_plot(
        doc,
        "event_study_cyclist_per_observed_day.png",
        "Figure 3. Cyclist event-study treated-control differences.",
    )

    add_heading(doc, "Next Step", 1)
    add_para(
        doc,
        "The next dissertation step is to draft the Methodology chapter using this final modelling position. The chapter should present the causal diagnostics as a strength of the workflow, while also explaining why the final claims remain exploratory rather than definitive.",
    )

    doc.save(OUT)


if __name__ == "__main__":
    build_doc()
    print(OUT)
